# Notebook B3: Hằng số phạt $P$ trong bài toán knapsack

Đi kèm [B-04: Hàm phạt và cách mã hóa](../../docs/B-extensions/04-penalties-and-encodings.md).

Ta lấy bài toán knapsack 8 vật, rồi biến thành QUBO bằng chính quy trình ở Mục 4 của bài báo (biến nhị phân, ràng buộc bất đẳng thức → đẳng thức bằng biến slack, hàm phạt $P(\cdot)^2$). Sau đó khảo sát: (a) $P$ nhỏ nhất để QUBO còn đúng, (b) $P$ tác động ra sao đến việc tìm nghiệm bằng simulated annealing, (c) so với công thức (36) của bài báo.

In [1]:
import itertools, sys, warnings
warnings.filterwarnings("ignore")
import numpy as np
import dimod
sys.path.insert(0, "../../code")
from qco_utils import integer_encoding

rng = np.random.default_rng(3)
n = 8
values  = rng.integers(3, 16, n)
weights = rng.integers(2, 10, n)
W = 20
print("giá trị :", values)
print("trọng lượng:", weights, "| sức chứa W =", W)

def feasible(x):  return sum(int(w) * xi for w, xi in zip(weights, x)) <= W
def total_value(x): return sum(int(v) * xi for v, xi in zip(values, x))
opt = max(total_value(x) for x in itertools.product((0, 1), repeat=n) if feasible(x))
print("giá trị tối ưu (vét cạn):", opt)

giá trị : [13  4  5  6  5 13 14 10]
trọng lượng: [2 2 4 5 6 5 4 3] | sức chứa W = 20
giá trị tối ưu (vét cạn): 59


## 1. Dựng QUBO

Cực tiểu hóa $-\sum_i v_i x_i + P\left(\sum_i w_i x_i + s - W\right)^2$, với slack $s \in [0, W]$ biểu diễn nhị phân bằng `integer_encoding` (Mục 4.1).

In [2]:
a, slack_w = integer_encoding(0, W)
print("trọng số của slack:", slack_w, "->", len(slack_w), "biến phụ (mã hóa nhị phân)")
xs = [f"x{i}" for i in range(n)]
ts = [f"s{j}" for j in range(len(slack_w))]

def build_qubo(P):
    bqm = dimod.BinaryQuadraticModel("BINARY")
    for i in range(n):
        bqm.add_variable(xs[i], -float(values[i]))
    terms = [(xs[i], float(weights[i])) for i in range(n)] + [(ts[j], float(slack_w[j])) for j in range(len(slack_w))]
    for u, c in terms:
        bqm.add_linear(u, P * (c*c - 2*W*c))
    for (u, c), (v, d) in itertools.combinations(terms, 2):
        bqm.add_quadratic(u, v, 2 * P * c * d)
    bqm.offset += P * W * W
    return bqm

def decode(sample):
    return [int(sample[v]) for v in xs]

trọng số của slack: [1, 2, 4, 8, 5] -> 5 biến phụ (mã hóa nhị phân)


## 2. $P$ nhỏ nhất để QUBO còn đúng

Với từng $P$, ta duyệt vét cạn cả 13 biến nhị phân ($2^{13} = 8192$ trạng thái) và kiểm tra: mọi trạng thái năng lượng thấp nhất có khả thi và có giá trị đúng bằng tối ưu hay không.

In [3]:
def exact_is_correct(P):
    ss = dimod.ExactSolver().sample(build_qubo(P))
    low = ss.lowest()
    return all(feasible(decode(s)) and total_value(decode(s)) == opt for s in low.samples())

grid = [0.25, 0.5, 1, 1.5, 2, 2.5, 3, 4, 5, 8, 16, 71, 160]
for P in grid:
    print(f"P = {P:6}: QUBO đúng (mọi nghiệm cực tiểu đều khả thi và tối ưu)? {exact_is_correct(P)}")

P =   0.25: QUBO đúng (mọi nghiệm cực tiểu đều khả thi và tối ưu)? False
P =    0.5: QUBO đúng (mọi nghiệm cực tiểu đều khả thi và tối ưu)? False
P =      1: QUBO đúng (mọi nghiệm cực tiểu đều khả thi và tối ưu)? False
P =    1.5: QUBO đúng (mọi nghiệm cực tiểu đều khả thi và tối ưu)? True
P =      2: QUBO đúng (mọi nghiệm cực tiểu đều khả thi và tối ưu)? True
P =    2.5: QUBO đúng (mọi nghiệm cực tiểu đều khả thi và tối ưu)? True
P =      3: QUBO đúng (mọi nghiệm cực tiểu đều khả thi và tối ưu)? True
P =      4: QUBO đúng (mọi nghiệm cực tiểu đều khả thi và tối ưu)? True
P =      5: QUBO đúng (mọi nghiệm cực tiểu đều khả thi và tối ưu)? True
P =      8: QUBO đúng (mọi nghiệm cực tiểu đều khả thi và tối ưu)? True
P =     16: QUBO đúng (mọi nghiệm cực tiểu đều khả thi và tối ưu)? True
P =     71: QUBO đúng (mọi nghiệm cực tiểu đều khả thi và tối ưu)? True
P =    160: QUBO đúng (mọi nghiệm cực tiểu đều khả thi và tối ưu)? True


In [4]:
for P in np.arange(0.1, 6.0, 0.1):
    if exact_is_correct(P):
        print(f"P nhỏ nhất (lưới 0.1) làm QUBO đúng: ~{P:.1f}")
        break
print("P theo công thức (36) áp dụng cho phần mục tiêu:", 1 + int(values.sum()))

P nhỏ nhất (lưới 0.1) làm QUBO đúng: ~1.1
P theo công thức (36) áp dụng cho phần mục tiêu: 71


Công thức (36) cho $P = 1 + \sum_i v_i$, lớn hơn rất nhiều so với mức tối thiểu thực tế. Giá trị này đúng (an toàn) nhưng không tối ưu. Câu hỏi tiếp theo là: $P$ lớn có hại không?

## 3. $P$ tác động thế nào đến việc tìm nghiệm bằng annealing?

Ta dùng một simulated annealing cổ điển tự viết (chạy song song nhiều lần bằng numpy) thay cho phần cứng. Lịch nhiệt độ **tự co giãn theo thang năng lượng của QUBO**, giống cách các solver thật làm: nhiệt độ đầu đủ cao để lật được spin tốn năng lượng lớn nhất, nhiệt độ cuối đủ thấp để phân biệt hệ số nhỏ nhất. Hệ quả: khi $P$ rất lớn, các hệ số lớn kéo nhiệt độ đầu lên cao, và phần lớn thời gian ủ bị lãng phí ở vùng quá nóng đối với phần mục tiêu thật.

Với mỗi $P$, ta đo (a) tỉ lệ lần chạy cho ra lời giải **khả thi**, (b) tỉ lệ cho ra lời giải **khả thi và tối ưu**, cả hai tính trên $x$ đã giải mã (bỏ biến slack).

In [5]:
coef = np.concatenate([weights, slack_w]).astype(float)
N = n + len(slack_w)

def qubo_arrays(P):
    h = np.zeros(N); h[:n] -= values
    h += P * (coef**2 - 2 * W * coef)
    J = 2 * P * np.outer(coef, coef); np.fill_diagonal(J, 0)      # E = sum_i h_i x_i + sum_{i<j} J_ij x_i x_j (J đối xứng)
    return h, J

def sa_run(P, sweeps, reads=2000, seed=0):
    r = np.random.default_rng(seed)
    h, J = qubo_arrays(P)
    max_dE = (np.abs(h) + np.abs(J).sum(1)).max()
    upper = np.abs(J[np.triu_indices(N, 1)])
    nonzero = np.concatenate([np.abs(h[h != 0]), upper[upper != 0]])
    betas = np.geomspace(np.log(2) / max_dE, np.log(100) / nonzero.min(), sweeps)
    X = r.integers(0, 2, size=(reads, N))
    for beta in betas:
        for i in range(N):
            dE = (1 - 2 * X[:, i]) * (h[i] + X @ J[:, i])
            accept = (dE < 0) | (r.random(reads) < np.exp(-beta * np.clip(dE, 0, None)))
            X[accept, i] ^= 1
    return X

def report(P, sweeps):
    X = sa_run(P, sweeps)
    sel = X[:, :n]
    feas_mask = (sel @ weights) <= W
    good_mask = feas_mask & ((sel @ values) == opt)
    h, J = qubo_arrays(P)
    scale = max(np.abs(h).max(), np.abs(J).max()) / values.min()
    return feas_mask.mean(), good_mask.mean(), scale

for sweeps in (100, 400):
    print(f"sweeps = {sweeps}")
    print("      P    khả thi   khả thi và tối ưu   thang năng lượng (max|hệ số| / giá trị nhỏ nhất)")
    for P in (0.5, 1, 1.5, 2, 4, 8, 16, 40, 71, 160, 1000):
        f_, g_, sc = report(P, sweeps)
        print(f"  {P:6}     {f_:5.2f}          {g_:5.2f}                 {sc:9.1f}")

sweeps = 100
      P    khả thi   khả thi và tối ưu   thang năng lượng (max|hệ số| / giá trị nhỏ nhất)


     0.5      0.45           0.42                      32.0


       1      0.59           0.25                      64.0
     1.5      0.74           0.15                      96.0


       2      0.82           0.10                     128.0


       4      0.95           0.04                     256.0


       8      0.98           0.01                     512.0
      16      0.98           0.01                    1024.0


... (10 outputs left out)

      40      0.99           0.00                    2560.0


      71      0.99           0.00                    4544.0


     160      0.99           0.00                   10240.0


    1000      0.99           0.00                   64000.0


Đọc kết quả (hai mức sweep cho cùng xu hướng):

- **Tỉ lệ khả thi tăng theo $P$**: ở $P = 0{,}5$ chưa đến một nửa lần chạy khả thi, còn từ $P \approx 8$ trở lên gần như luôn khả thi. Ràng buộc càng bị phạt nặng thì càng được tuân thủ.
- **Tỉ lệ "khả thi và tối ưu" giảm đơn điệu theo $P$**: từ khoảng 0,25–0,45 ở $P \lesssim 1$ (tùy số sweep) xuống gần 0 từ $P \approx 16$. Lý do: khi $P$ lớn, bộ ủ vừa tốn nhiệt độ vào phần quá nóng, vừa bị giam trong một vùng khả thi mà các nghiệm khả thi khác nhau cách nhau bởi rào cản năng lượng rất cao, nên khó chuyển từ nghiệm khả thi kém sang nghiệm khả thi tốt hơn.
- Điểm tốt nhất nằm **gần ngưỡng $P^* \approx 1{,}1$** tìm được ở Mục 2 (thậm chí hơi dưới ngưỡng, vì một số lần chạy không cần QUBO đúng để cho lời giải khả thi tốt). Giá trị $P = 71$ của công thức (36) cho gần như 0% thành công trong thí nghiệm này, dù QUBO tương ứng hoàn toàn đúng về mặt toán học.

Cần lưu ý: đây là một thể hiện knapsack nhỏ và một sampler cổ điển, nên số cụ thể sẽ khác trên phần cứng và bài toán khác. Điều đáng mang đi là **bản chất đánh đổi** (khả thi so với chất lượng) và việc công thức (36) là cận an toàn về tính đúng đắn của QUBO, không phải lựa chọn tốt về hiệu năng.

## 4. Số biến theo cách mã hóa slack

Cùng ràng buộc $\sum w_i x_i \le W$, slack nhị phân dùng ít qubit nhất; cách mã hóa unary (mỗi đơn vị một biến) dùng nhiều hơn nhiều.

In [6]:
for W_ in (10, 20, 50, 100, 500, 1000):
    a_, w_ = integer_encoding(0, W_)
    print(f"W = {W_:5d}: slack nhị phân cần {len(w_):2d} biến | slack unary cần {W_:5d} biến")

W =    10: slack nhị phân cần  4 biến | slack unary cần    10 biến
W =    20: slack nhị phân cần  5 biến | slack unary cần    20 biến
W =    50: slack nhị phân cần  6 biến | slack unary cần    50 biến
W =   100: slack nhị phân cần  7 biến | slack unary cần   100 biến
W =   500: slack nhị phân cần  9 biến | slack unary cần   500 biến
W =  1000: slack nhị phân cần 10 biến | slack unary cần  1000 biến
